In [ ]:
# ABOUTME: Interactive notebook teaching classification metrics through real-world metaphors
# ABOUTME: Covers accuracy, precision, recall, F1-score and F-beta with dedicated widgets for each
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import confusion_matrix
from ipywidgets import interact, widgets
%matplotlib widget

# Classification Metrics: A Survival Guide

*"Not everything that counts can be counted, and not everything that can be counted counts."* — William Bruce Cameron

In classification, the numbers you choose to look at determine the story you tell. This notebook teaches you to read between the metrics.

## The Confusion Matrix — Your Scorecard

```
                  PREDICTED
                  Negative    Positive
ACTUAL  Negative  [ TN ]      [ FP ]
        Positive  [ FN ]      [ TP ]
```

**Mnemonic:** The **second word** tells you what the model **said**. The **first word** tells you whether it was **right**.

- **True Positive:** model said Positive, and it was Right (True)
- **False Positive:** model said Positive, and it was Wrong (False)
- **True Negative:** model said Negative, and it was Right
- **False Negative:** model said Negative, and it was Wrong

## Part 1: Accuracy — The Misleading Metric

### The Airport Security Scanner

Imagine you're running airport security. Every day, 10,000 bags go through your scanner.

On average, only **10 bags** contain something dangerous. The other 9,990 are perfectly safe.

Your boss gives you a 'state-of-the-art' AI scanner. It predicts *every single bag* as safe.

How accurate is it?

In [ ]:
fig_airport, axes_airport = plt.subplots(1, 2, figsize=(10, 4))

n_bags = 10000
n_dangerous = 10
y_true_bags = np.array([1] * n_dangerous + [0] * (n_bags - n_dangerous))
y_pred_lazy = np.zeros(n_bags, dtype=int)  # Predict all safe

cm_bags = confusion_matrix(y_true_bags, y_pred_lazy, labels=[0, 1])
TN, FP, FN, TP = cm_bags.ravel()

accuracy_bags = (TP + TN) / n_bags

# Left: Confusion matrix
ax = axes_airport[0]
sns.heatmap(cm_bags, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Safe', 'Dangerous'], yticklabels=['Safe', 'Dangerous'],
            cbar=False, annot_kws={'size': 14})
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

# Right: Big accuracy number
ax2 = axes_airport[1]
ax2.axis('off')
ax2.text(0.5, 0.65, f'{accuracy_bags:.1%}', fontsize=60, fontweight='bold',
         ha='center', va='center', color='forestgreen', transform=ax2.transAxes)
ax2.text(0.5, 0.35, 'Accuracy', fontsize=20, ha='center', va='center', transform=ax2.transAxes)
ax2.text(0.5, 0.15, f'...but {n_dangerous} dangerous bags got through.',
         fontsize=13, ha='center', va='center', color='crimson', transform=ax2.transAxes)

fig_airport.suptitle('The "Everything is Safe" Scanner', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**The punchline:** This 'model' is 99.9% accurate, but it's completely useless.

**Rule of thumb:** Before celebrating accuracy, always ask: *What would a dumb baseline achieve?*

### Accuracy — The Full Picture

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{\text{Correct predictions}}{\text{All predictions}}$$

**Intuition:** How often is the model right, overall?

**When to use:** Balanced datasets where both classes matter equally.

**Limitation:** Useless on imbalanced datasets. A model that always predicts the majority class gets high accuracy for free.

In [ ]:
fig_acc, axes_acc = plt.subplots(1, 2, figsize=(9, 4))

@interact(
    TP=widgets.IntSlider(min=0, max=100, value=40, description='TP:'),
    TN=widgets.IntSlider(min=0, max=100, value=50, description='TN:'),
    FP=widgets.IntSlider(min=0, max=100, value=5, description='FP:'),
    FN=widgets.IntSlider(min=0, max=100, value=5, description='FN:'),
)
def build_confusion_matrix(TP, TN, FP, FN):
    for ax in axes_acc:
        ax.clear()
    
    total = TP + TN + FP + FN
    accuracy = (TP + TN) / total if total > 0 else 0
    
    cm = np.array([[TN, FP], [FN, TP]])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_acc[0],
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'],
                cbar=False, annot_kws={'size': 16})
    axes_acc[0].set_xlabel('Predicted')
    axes_acc[0].set_ylabel('Actual')
    axes_acc[0].set_title('Confusion Matrix')
    
    # Accuracy bar
    axes_acc[1].barh(['Accuracy'], [accuracy], color='steelblue', height=0.4)
    axes_acc[1].barh(['Accuracy'], [1.0], color='lightgray', height=0.4, alpha=0.3)
    axes_acc[1].set_xlim(0, 1.05)
    axes_acc[1].text(accuracy + 0.02, 0, f'{accuracy:.1%}', va='center', fontsize=14, fontweight='bold')
    axes_acc[1].set_title('Accuracy')
    
    fig_acc.suptitle(f'Build Your Own Confusion Matrix (n={total})', fontsize=13, fontweight='bold')
    fig_acc.tight_layout()
    fig_acc.canvas.draw_idle()

Now that we see accuracy's blind spot, let's look at metrics that tell a more nuanced story.

## Part 2: Precision — The Purity of Your Predictions

### The Fisherman's Net

You are a fisherman. You cast your net into the sea looking for tuna.

When you pull it up, the net contains: tuna (what you wanted), jellyfish, seaweed, an old boot...

**Precision** answers: *Of everything in my net, how much is actually tuna?*

### Precision — The Formula

$$\text{Precision} = \frac{TP}{TP + FP} = \frac{\text{True catches}}{\text{Everything the model flagged}}$$

**Intuition:** When the model says 'Positive', how often is it right?

**Use when the cost of False Positives is high:**
- Cancer treatment: you don't want to put a healthy person through chemotherapy (FP)
- Criminal conviction: you don't want to jail an innocent person (FP)
- Spam filter: you don't want important emails in the spam folder (FP)

**Limitation:** Precision is blind to False Negatives. A model that only makes ONE prediction (and gets it right) has 100% precision.

In [ ]:
fig_fish, axes_fish = plt.subplots(1, 2, figsize=(10, 5))

@interact(
    tuna_caught=widgets.IntSlider(min=0, max=30, value=15, description='Tuna caught (TP):',
                                   style={'description_width': 'initial'}),
    junk_caught=widgets.IntSlider(min=0, max=30, value=5, description='Junk caught (FP):',
                                   style={'description_width': 'initial'}),
    tuna_missed=widgets.IntSlider(min=0, max=30, value=8, description='Tuna missed (FN):',
                                   style={'description_width': 'initial'}),
)
def fisherman_net(tuna_caught, junk_caught, tuna_missed):
    for ax in axes_fish:
        ax.clear()
    
    ax = axes_fish[0]
    
    # Draw the net (circle)
    net_circle = plt.Circle((0.5, 0.5), 0.35, fill=False, edgecolor='saddlebrown', 
                             linewidth=3, linestyle='--')
    ax.add_patch(net_circle)
    ax.text(0.5, 0.88, 'THE NET', ha='center', fontsize=11, fontweight='bold', color='saddlebrown')
    
    rng = np.random.RandomState(42)
    
    # Tuna inside net (TP) - green fish markers
    if tuna_caught > 0:
        angles = rng.uniform(0, 2*np.pi, tuna_caught)
        radii = rng.uniform(0, 0.30, tuna_caught)
        x_tp = 0.5 + radii * np.cos(angles)
        y_tp = 0.5 + radii * np.sin(angles)
        ax.scatter(x_tp, y_tp, c='forestgreen', s=80, marker='>', alpha=0.8, 
                  edgecolors='darkgreen', linewidth=1, zorder=3)
    
    # Junk inside net (FP) - red X markers
    if junk_caught > 0:
        angles = rng.uniform(0, 2*np.pi, junk_caught)
        radii = rng.uniform(0, 0.30, junk_caught)
        x_fp = 0.5 + radii * np.cos(angles)
        y_fp = 0.5 + radii * np.sin(angles)
        ax.scatter(x_fp, y_fp, c='indianred', s=80, marker='X', alpha=0.8,
                  edgecolors='darkred', linewidth=1, zorder=3)
    
    # Tuna outside net (FN) - gray fish markers
    if tuna_missed > 0:
        angles = rng.uniform(0, 2*np.pi, tuna_missed)
        radii = rng.uniform(0.38, 0.48, tuna_missed)
        x_fn = 0.5 + radii * np.cos(angles)
        y_fn = 0.5 + radii * np.sin(angles)
        ax.scatter(x_fn, y_fn, c='gray', s=80, marker='>', alpha=0.5,
                  edgecolors='dimgray', linewidth=1, zorder=3)
    
    patches = [
        mpatches.Patch(color='forestgreen', label=f'Tuna caught (TP): {tuna_caught}'),
        mpatches.Patch(color='indianred', label=f'Junk caught (FP): {junk_caught}'),
        mpatches.Patch(color='gray', label=f'Tuna missed (FN): {tuna_missed}'),
    ]
    ax.legend(handles=patches, loc='lower left', fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_title("The Fisherman's Net")
    ax.axis('off')
    
    # Right: Precision and Recall bars
    ax2 = axes_fish[1]
    precision = tuna_caught / (tuna_caught + junk_caught) if (tuna_caught + junk_caught) > 0 else 0
    recall = tuna_caught / (tuna_caught + tuna_missed) if (tuna_caught + tuna_missed) > 0 else 0
    
    bars = ax2.barh(['Precision', 'Recall'], [precision, recall], 
                    color=['steelblue', 'darkorange'], height=0.5)
    ax2.barh(['Precision', 'Recall'], [1.0, 1.0], color='lightgray', height=0.5, alpha=0.3)
    ax2.set_xlim(0, 1.15)
    for bar, val in zip(bars, [precision, recall]):
        ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2, 
                f'{val:.1%}', va='center', fontsize=13, fontweight='bold')
    ax2.set_title('Metrics')
    
    fig_fish.suptitle("Precision: How pure is what's in your net?", fontsize=13, fontweight='bold')
    fig_fish.tight_layout()
    fig_fish.canvas.draw_idle()

### Try it yourself:

1. Set `Junk caught = 0`. What happens to precision? (Hint: it's perfect... but is the net useful?)
2. Now set `Tuna missed = 20`. Precision stays the same! **Precision doesn't care about what you missed.**
3. This is precision's blind spot: it only looks inside the net, not at what escaped.

## Part 3: Recall — Did You Find Them All?

### The Cancer Screening

You run a hospital screening program. 1,000 patients come in for a cancer test.

50 of them actually have cancer. Your test flags some as positive.

**Recall** answers: *Of all the people who actually have cancer, how many did my test catch?*

In medicine, a False Negative can mean a patient goes home thinking they're healthy — and the cancer grows until it's too late.

**False Negatives kill.**

### Recall — The Formula

$$\text{Recall} = \frac{TP}{TP + FN} = \frac{\text{Caught}}{\text{All actual positives}}$$

Also called **Sensitivity** or **True Positive Rate (TPR)**.

**Intuition:** Of all the real positives, how many did I find?

**Use when the cost of False Negatives is high:**
- Cancer screening: missing a cancer = death
- Fraud detection: missing fraud = financial loss
- Airport security: missing a threat = catastrophe

**Limitation:** Recall is blind to False Positives. A model that predicts *everything* as positive has 100% recall.

In [ ]:
fig_grid, axes_grid = plt.subplots(1, 2, figsize=(10, 5))

@interact(
    n_patients=widgets.IntSlider(min=20, max=200, step=10, value=100, description='Patients:',
                                  style={'description_width': 'initial'}),
    n_sick=widgets.IntSlider(min=1, max=50, value=10, description='Sick:',
                              style={'description_width': 'initial'}),
    n_caught=widgets.IntSlider(min=0, max=50, value=7, description='Caught (TP):',
                                style={'description_width': 'initial'}),
    n_false_alarms=widgets.IntSlider(min=0, max=50, value=5, description='False alarms (FP):',
                                      style={'description_width': 'initial'}),
)
def patient_grid(n_patients, n_sick, n_caught, n_false_alarms):
    for ax in axes_grid:
        ax.clear()
    
    # Clamp values to valid ranges
    n_sick = min(n_sick, n_patients)
    n_caught = min(n_caught, n_sick)
    n_healthy = n_patients - n_sick
    n_false_alarms = min(n_false_alarms, n_healthy)
    
    TP = n_caught
    FN = n_sick - n_caught
    FP = n_false_alarms
    TN = n_healthy - n_false_alarms
    
    ax = axes_grid[0]
    # Build grid
    cols = int(np.ceil(np.sqrt(n_patients)))
    rows = int(np.ceil(n_patients / cols))
    
    # Assign categories: first n_sick are sick (TP then FN), rest are healthy (FP then TN)
    for i in range(n_patients):
        row = i // cols
        col = i % cols
        
        if i < TP:
            color = 'forestgreen'     # TP: sick and caught
        elif i < TP + FN:
            color = 'crimson'         # FN: sick and missed
        elif i < TP + FN + FP:
            color = 'orange'          # FP: healthy but flagged
        else:
            color = 'steelblue'       # TN: healthy and correctly ignored
        
        rect = plt.Rectangle((col, rows - row - 1), 0.85, 0.85, 
                             facecolor=color, edgecolor='white', linewidth=0.5)
        ax.add_patch(rect)
    
    ax.set_xlim(-0.1, cols + 0.1)
    ax.set_ylim(-0.1, rows + 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    patches = [
        mpatches.Patch(color='forestgreen', label=f'Caught sick (TP): {TP}'),
        mpatches.Patch(color='crimson', label=f'Missed sick (FN): {FN}'),
        mpatches.Patch(color='orange', label=f'False alarm (FP): {FP}'),
        mpatches.Patch(color='steelblue', label=f'Correct healthy (TN): {TN}'),
    ]
    ax.legend(handles=patches, loc='upper right', fontsize=8, framealpha=0.9)
    ax.set_title('Patient Population')
    
    # Right: metric bars
    ax2 = axes_grid[1]
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    
    bars = ax2.barh(['Recall', 'Precision'], [recall, precision],
                    color=['darkorange', 'steelblue'], height=0.5)
    ax2.barh(['Recall', 'Precision'], [1.0, 1.0], color='lightgray', height=0.5, alpha=0.3)
    ax2.set_xlim(0, 1.15)
    for bar, val in zip(bars, [recall, precision]):
        ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.1%}', va='center', fontsize=13, fontweight='bold')
    ax2.set_title('Metrics')
    
    fig_grid.suptitle(f'Cancer Screening: {n_patients} patients, {n_sick} sick', 
                      fontsize=13, fontweight='bold')
    fig_grid.tight_layout()
    fig_grid.canvas.draw_idle()

### Try it yourself:

1. Set `Caught = n_sick` (catch all sick patients). Recall = 100%! But watch what happens to precision if false alarms are high.
2. Now set `Caught = 0`. Recall drops to 0% — every sick patient goes home undiagnosed.
3. **Precision and Recall are complementary blind spots.** Precision ignores what you missed; Recall ignores the false alarms.

## Part 4: The Precision–Recall Trade-off

Imagine casting a wider net: you catch more tuna (recall goes up), but also more junk (precision goes down).

A tighter net: less junk (precision up), but tuna escape (recall down).

This isn't a bug — it's the fundamental tension of classification.

In [ ]:
np.random.seed(42)
n_pos = 50
n_neg = 200
scores_pos = np.random.beta(5, 2, n_pos)    # Positive class: skewed toward 1
scores_neg = np.random.beta(2, 5, n_neg)    # Negative class: skewed toward 0
scores = np.concatenate([scores_neg, scores_pos])
y_true_synth = np.concatenate([np.zeros(n_neg), np.ones(n_pos)])

fig_tradeoff, axes_tradeoff = plt.subplots(1, 2, figsize=(11, 4.5))

@interact(
    threshold=widgets.FloatSlider(
        min=0.0, max=1.0, step=0.02, value=0.5,
        description='Threshold:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='450px')
    )
)
def precision_recall_tradeoff(threshold):
    for ax in axes_tradeoff:
        ax.clear()
    
    y_pred = (scores >= threshold).astype(int)
    
    # Left: score scatter
    ax = axes_tradeoff[0]
    ax.axvline(x=threshold, color='black', linestyle='--', linewidth=2)
    ax.axvspan(0, threshold, alpha=0.08, color='steelblue')
    ax.axvspan(threshold, 1, alpha=0.08, color='indianred')
    
    jitter_rng = np.random.RandomState(0)
    jitter = jitter_rng.uniform(-0.3, 0.3, len(scores))
    
    neg_mask = y_true_synth == 0
    pos_mask = y_true_synth == 1
    ax.scatter(scores[neg_mask], jitter[neg_mask], c='steelblue', alpha=0.5, s=25, label='Negative')
    ax.scatter(scores[pos_mask], jitter[pos_mask], c='indianred', alpha=0.5, s=25, label='Positive')
    ax.set_xlabel('Model Score')
    ax.set_yticks([])
    ax.set_title(f'Threshold = {threshold:.2f}')
    ax.legend(fontsize=9)
    ax.set_xlim(-0.05, 1.05)
    
    # Right: metric bars
    ax2 = axes_tradeoff[1]
    cm = confusion_matrix(y_true_synth, y_pred, labels=[0, 1])
    TN, FP, FN, TP = cm.ravel()
    
    accuracy = (TP + TN) / len(y_true_synth) if len(y_true_synth) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
    values = [accuracy, precision, recall, f1]
    colors = ['gray', 'steelblue', 'darkorange', 'forestgreen']
    
    bars = ax2.barh(metrics, values, color=colors, height=0.6)
    ax2.barh(metrics, [1]*4, color='lightgray', height=0.6, alpha=0.2)
    ax2.set_xlim(0, 1.15)
    for bar, val in zip(bars, values):
        ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.1%}', va='center', fontsize=11, fontweight='bold')
    ax2.set_title('Metrics')
    
    fig_tradeoff.suptitle('Precision\u2013Recall Trade-off', fontsize=14, fontweight='bold')
    fig_tradeoff.tight_layout()
    fig_tradeoff.canvas.draw_idle()

**You cannot maximize both at once.** Choosing a metric means choosing what kind of mistakes you're willing to tolerate.

## Part 5: F1-Score — The Diplomat

### Why Not Just Average?

If precision = 0.9 and recall = 0.1, the arithmetic mean is 0.50 — not terrible?

But a model with 90% precision and 10% recall is *useless*: it barely catches any positives!

The **harmonic mean** is much harsher on imbalanced values. That's the F1-score.

In [ ]:
fig_f1, axes_f1 = plt.subplots(1, 2, figsize=(11, 4.5))

@interact(
    precision=widgets.FloatSlider(min=0.01, max=1.0, step=0.01, value=0.9, description='Precision:',
                                    style={'description_width': 'initial'}),
    recall=widgets.FloatSlider(min=0.01, max=1.0, step=0.01, value=0.1, description='Recall:',
                                style={'description_width': 'initial'}),
)
def harmonic_vs_arithmetic(precision, recall):
    for ax in axes_f1:
        ax.clear()
    
    arithmetic = (precision + recall) / 2
    harmonic = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Left: bar comparison
    ax = axes_f1[0]
    x_pos = np.arange(4)
    vals = [precision, recall, arithmetic, harmonic]
    colors = ['steelblue', 'darkorange', 'mediumpurple', 'forestgreen']
    labels_bar = ['Precision', 'Recall', 'Arithmetic\nMean', 'Harmonic\nMean (F1)']
    
    bars = ax.bar(x_pos, vals, color=colors, width=0.6, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}', 
               ha='center', fontsize=11, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels_bar, fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_title('Comparison')
    
    # Right: F1 as function of recall (with precision fixed)
    ax2 = axes_f1[1]
    recall_range = np.linspace(0.01, 1.0, 100)
    f1_curve = 2 * precision * recall_range / (precision + recall_range)
    am_curve = (precision + recall_range) / 2
    
    ax2.plot(recall_range, f1_curve, 'forestgreen', linewidth=2, label='Harmonic Mean (F1)')
    ax2.plot(recall_range, am_curve, 'mediumpurple', linewidth=2, linestyle='--', label='Arithmetic Mean')
    ax2.axvline(x=recall, color='darkorange', linestyle=':', alpha=0.7)
    ax2.plot(recall, harmonic, 'go', markersize=10, zorder=5)
    ax2.plot(recall, arithmetic, 'ms', markersize=10, zorder=5)
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Mean Value')
    ax2.set_title(f'Means with Precision = {precision:.2f}')
    ax2.legend(fontsize=9)
    ax2.set_ylim(0, 1.05)
    
    fig_f1.suptitle('Harmonic Mean is Harsher Than Arithmetic Mean', fontsize=13, fontweight='bold')
    fig_f1.tight_layout()
    fig_f1.canvas.draw_idle()

### F1-Score — The Formula

$$F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Intuition:** F1 is the harmonic mean of precision and recall. It punishes extreme imbalances — if either is low, F1 will be low.

**When to use:** When you need a single number to compare models and care about both precision and recall.

**Limitation:** F1 treats precision and recall as equally important. But what if one matters more than the other?

### F-beta: Tuning the Balance

$$F_\beta = (1 + \beta^2) \times \frac{\text{Precision} \times \text{Recall}}{(\beta^2 \times \text{Precision}) + \text{Recall}}$$

$\beta$ controls which metric matters more:

- $\beta = 1$: F1 — precision and recall weighted equally
- $\beta = 0.5$: F0.5 — **precision matters more** (e.g., spam filter)
- $\beta = 2$: F2 — **recall matters more** (e.g., cancer screening)

Think of $\beta$ as how many times more important recall is than precision.

In [ ]:
fig_fbeta, axes_fbeta = plt.subplots(1, 2, figsize=(11, 5))

@interact(
    precision=widgets.FloatSlider(min=0.01, max=1.0, step=0.01, value=0.7, description='Precision:',
                                    style={'description_width': 'initial'}),
    recall=widgets.FloatSlider(min=0.01, max=1.0, step=0.01, value=0.4, description='Recall:',
                                style={'description_width': 'initial'}),
    beta=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Beta:',
                              style={'description_width': 'initial'}),
)
def explore_fbeta(precision, recall, beta):
    for ax in axes_fbeta:
        ax.clear()
    
    def fbeta(p, r, b):
        if (b**2 * p + r) == 0:
            return 0
        return (1 + b**2) * p * r / (b**2 * p + r)
    
    f05 = fbeta(precision, recall, 0.5)
    f1 = fbeta(precision, recall, 1.0)
    f2 = fbeta(precision, recall, 2.0)
    f_custom = fbeta(precision, recall, beta)
    
    # Left: comparison bars
    ax = axes_fbeta[0]
    labels_bar = ['Precision', 'Recall', f'F0.5', 'F1', 'F2', f'F{beta:.1f}']
    vals = [precision, recall, f05, f1, f2, f_custom]
    colors = ['steelblue', 'darkorange', '#9b59b6', 'forestgreen', '#e74c3c', '#f39c12']
    
    bars = ax.barh(labels_bar, vals, color=colors, height=0.55)
    ax.barh(labels_bar, [1]*len(labels_bar), color='lightgray', height=0.55, alpha=0.2)
    ax.set_xlim(0, 1.15)
    for bar, val in zip(bars, vals):
        ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
               f'{val:.2f}', va='center', fontsize=10, fontweight='bold')
    ax.set_title('F-scores Comparison')
    
    # Right: F-beta as function of beta
    ax2 = axes_fbeta[1]
    beta_range = np.linspace(0.1, 3.0, 100)
    fbeta_curve = [(1 + b**2) * precision * recall / (b**2 * precision + recall) 
                   if (b**2 * precision + recall) > 0 else 0 
                   for b in beta_range]
    
    ax2.plot(beta_range, fbeta_curve, 'forestgreen', linewidth=2)
    ax2.axhline(y=precision, color='steelblue', linestyle='--', alpha=0.5, label='Precision')
    ax2.axhline(y=recall, color='darkorange', linestyle='--', alpha=0.5, label='Recall')
    ax2.axvline(x=beta, color='#f39c12', linestyle=':', alpha=0.7)
    ax2.plot(beta, f_custom, 'o', color='#f39c12', markersize=10, zorder=5)
    ax2.set_xlabel(r'$\beta$')
    ax2.set_ylabel(r'$F_\beta$')
    ax2.set_title(r'$F_\beta$ as a function of $\beta$')
    ax2.legend(fontsize=9)
    ax2.set_ylim(0, 1.05)
    
    fig_fbeta.suptitle(r'F-beta: Tuning the Precision\u2013Recall Balance', fontsize=13, fontweight='bold')
    fig_fbeta.tight_layout()
    fig_fbeta.canvas.draw_idle()

## Part 6: The Full Dashboard

Let's bring everything together. This dashboard shows all metrics at once for a threshold-based classifier.

In [ ]:
# Reuse the synthetic data from the trade-off section
fig_dash, axes_dash = plt.subplots(2, 2, figsize=(12, 8))
fig_dash.subplots_adjust(hspace=0.4, wspace=0.3)

@interact(
    threshold=widgets.FloatSlider(min=0.0, max=1.0, step=0.02, value=0.5, description='Threshold:',
                                    style={'description_width': 'initial'},
                                    layout=widgets.Layout(width='450px')),
    beta=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Beta:',
                              style={'description_width': 'initial'},
                              layout=widgets.Layout(width='450px')),
)
def full_dashboard(threshold, beta):
    for ax in axes_dash.flat:
        ax.clear()
    
    y_pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true_synth, y_pred, labels=[0, 1])
    TN, FP, FN, TP = cm.ravel()
    
    total = TP + TN + FP + FN
    accuracy = (TP + TN) / total if total > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    f_beta = (1 + beta**2) * precision * recall / (beta**2 * precision + recall) if (beta**2 * precision + recall) > 0 else 0
    
    # Panel 1: Score scatter
    ax1 = axes_dash[0, 0]
    jitter_rng = np.random.RandomState(0)
    jitter = jitter_rng.uniform(-0.3, 0.3, len(scores))
    ax1.axvline(x=threshold, color='black', linestyle='--', linewidth=2)
    ax1.axvspan(0, threshold, alpha=0.08, color='steelblue')
    ax1.axvspan(threshold, 1, alpha=0.08, color='indianred')
    neg_mask = y_true_synth == 0
    pos_mask = y_true_synth == 1
    ax1.scatter(scores[neg_mask], jitter[neg_mask], c='steelblue', alpha=0.5, s=20)
    ax1.scatter(scores[pos_mask], jitter[pos_mask], c='indianred', alpha=0.5, s=20)
    ax1.set_xlabel('Score')
    ax1.set_yticks([])
    ax1.set_title(f'Threshold = {threshold:.2f}')
    ax1.set_xlim(-0.05, 1.05)
    
    # Panel 2: Confusion matrix
    ax2 = axes_dash[0, 1]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'],
                cbar=False, annot_kws={'size': 16})
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('Actual')
    ax2.set_title('Confusion Matrix')
    
    # Panel 3: All metrics bars
    ax3 = axes_dash[1, 0]
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', f'F{beta:.1f}']
    metric_vals = [accuracy, precision, recall, f1, f_beta]
    metric_colors = ['gray', 'steelblue', 'darkorange', 'forestgreen', '#f39c12']
    
    bars = ax3.barh(metric_names, metric_vals, color=metric_colors, height=0.55)
    ax3.barh(metric_names, [1]*len(metric_names), color='lightgray', height=0.55, alpha=0.2)
    ax3.set_xlim(0, 1.15)
    for bar, val in zip(bars, metric_vals):
        ax3.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.1%}', va='center', fontsize=10, fontweight='bold')
    ax3.set_title('All Metrics')
    
    # Panel 4: Outcome counts
    ax4 = axes_dash[1, 1]
    categories = ['TN', 'TP', 'FP', 'FN']
    values = [TN, TP, FP, FN]
    bar_colors = ['steelblue', 'forestgreen', 'orange', 'crimson']
    b = ax4.bar(categories, values, color=bar_colors, edgecolor='white')
    for bar, val in zip(b, values):
        ax4.text(bar.get_x() + bar.get_width()/2, val + 1,
                str(val), ha='center', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Count')
    ax4.set_title('Outcome Breakdown')
    ax4.set_ylim(0, max(values) + 15 if max(values) > 0 else 10)
    
    fig_dash.suptitle('Classification Dashboard', fontsize=14, fontweight='bold')
    fig_dash.canvas.draw_idle()

## Cheat Sheet

| Metric | Formula | What it cares about | Blind spot | When to use |
|--------|---------|--------------------|-----------|----||
| **Accuracy** | $\frac{TP+TN}{All}$ | Overall correctness | Ignores class imbalance | Balanced datasets |
| **Precision** | $\frac{TP}{TP+FP}$ | Purity of predictions | Ignores FN (missed positives) | Cost of FP is high (spam, conviction) |
| **Recall** | $\frac{TP}{TP+FN}$ | Completeness of detection | Ignores FP (false alarms) | Cost of FN is high (cancer, fraud, security) |
| **F1** | $\frac{2 \cdot P \cdot R}{P + R}$ | Balance of P and R | Treats both equally | Single comparison number |
| **F-beta** | See formula above | Weighted P and R | You must choose beta | Domain-specific needs |

## Going Further

- **Balanced Accuracy**: Average of recall per class — handles imbalance
- **ROC-AUC**: Area under the ROC curve — threshold-independent ranking metric
- **PR-AUC**: Area under the Precision-Recall curve — better for very imbalanced data
- **Cohen's Kappa**: Accuracy adjusted for chance agreement

The right metric depends on the right question. Always start by asking: *What is the cost of being wrong, and in which direction?*